In [ ]:
import pandas as pd

df = pd.read_csv("scalabilityExperimentMSDv2.csv")
df = pd.read_csv("partial/scalabilityExperimentcVaR_partial.csv")

In [ ]:
display(df)
plot_df = df.copy()

plot_df["method_small"] = "small_support"
plot_df["method_mcp"] = "path_MCP"
plot_df["problem_size"] = plot_df["K"] * plot_df["n"] * plot_df["n"]

summary = plot_df.groupby(["K", "n", "kappa", "tau"], as_index=False, dropna=False).agg(
    reps=("seed", "count"),
    problem_size=("problem_size", "first"),
    small_success_rate=("small_success", "mean"),
    mcp_success_rate=("mcp_success", "mean"),
    small_time_median=("small_time_s", "median"),
    mcp_time_median=("mcp_time_s", "median"),
    small_time_q25=("small_time_s", lambda x: x.quantile(0.25)),
    small_time_q75=("small_time_s", lambda x: x.quantile(0.75)),
    mcp_time_q25=("mcp_time_s", lambda x: x.quantile(0.25)),
    mcp_time_q75=("mcp_time_s", lambda x: x.quantile(0.75)),
    small_eta_median=("small_eta", "median"),
    mcp_eta_median=("mcp_eta", "median"),
)

small_summary = (
    plot_df[plot_df["small_success"]]
    .groupby(["K", "n", "kappa", "tau"], as_index=False)
    .agg(
        reps_succ_small=("seed", "count"),
        problem_size=("problem_size", "first"),
        small_time_median=("small_time_s", "median"),
        small_time_q25=("small_time_s", lambda x: x.quantile(0.25)),
        small_time_q75=("small_time_s", lambda x: x.quantile(0.75)),
        small_eta_median=("small_eta", "median"),
    )
)

mcp_summary = (
    plot_df[plot_df["mcp_success"]]
    .groupby(["K", "n", "kappa", "tau"], as_index=False)
    .agg(
        reps_succ_mcp=("seed", "count"),
        problem_size=("problem_size", "first"),
        mcp_time_median=("mcp_time_s", "median"),
        mcp_time_q25=("mcp_time_s", lambda x: x.quantile(0.25)),
        mcp_time_q75=("mcp_time_s", lambda x: x.quantile(0.75)),
        mcp_eta_median=("mcp_eta", "median"),
    )
)

success_summary = mcp_summary.merge(
    small_summary, on=["K", "n", "kappa", "tau", "problem_size"], how="outer"
)
display(success_summary)
display(summary)

In [ ]:
import matplotlib.pyplot as plt

K_grid = [5, 10, 30, 100, 250, 500]
n_grid = [5, 10, 20, 50]

small_color = "tab:blue"
mcp_color = "tab:orange"
fig, axes = plt.subplots(1, len(n_grid), figsize=(15, 3.5), dpi=150, sharey=True)

for ax, n in zip(axes, n_grid):
    s = success_summary[success_summary["n"] == n].sort_values("K")

    ax.plot(s["K"], s["small_time_median"], "o-", label="small", color=small_color)
    ax.fill_between(
        s["K"],
        s["small_time_q25"],
        s["small_time_q75"],
        color=small_color,
        alpha=0.15,
        label="_nolegend_",
    )

    ax.plot(s["K"], s["mcp_time_median"], "s--", label="MCP", color=mcp_color)
    ax.fill_between(
        s["K"],
        s["mcp_time_q25"],
        s["mcp_time_q75"],
        color=mcp_color,
        alpha=0.15,
        label="_nolegend_",
    )

    ax.set_yscale("log")
    ax.set_title(f"n={n}")
    ax.set_xlabel("K")
    ax.grid(alpha=0.25)

axes[0].set_ylabel("Median runtime (s)")
axes[-1].legend(frameon=False)
plt.tight_layout()
plt.show()

# remove header
# add no successful runs and legend
# describe the setup for the experiment and the hardware

In [ ]:
import matplotlib.pyplot as plt

K_grid = [5, 10, 30, 100, 250, 500]
n_grid = [5, 10, 20, 50]

cmap = plt.get_cmap("inferno")
small_color = cmap(0.25)
mcp_color = cmap(0.75)

fig, axes = plt.subplots(
    1,
    len(n_grid),
    figsize=(15, 3.5),
    dpi=150,
    sharey=True,
)

for ax, n in zip(axes, n_grid):
    s = summary[summary["n"] == n].sort_values("K")

    small_zero = s["small_success_rate"] == 0
    mcp_zero = s["mcp_success_rate"] == 0

    # Lines only, no markers
    ax.plot(
        s["K"],
        s["small_time_median"],
        "-",
        color=small_color,
        linewidth=1.8,
        label="small-support" if n == n_grid[0] else "_nolegend_",
    )
    ax.plot(
        s["K"],
        s["mcp_time_median"],
        "--",
        color=mcp_color,
        linewidth=1.8,
        label="MCP" if n == n_grid[0] else "_nolegend_",
    )

    # IQR bands
    ax.fill_between(
        s["K"],
        s["small_time_q25"],
        s["small_time_q75"],
        color=small_color,
        alpha=0.15,
    )
    ax.fill_between(
        s["K"],
        s["mcp_time_q25"],
        s["mcp_time_q75"],
        color=mcp_color,
        alpha=0.15,
    )

    # Normal markers only where success rate > 0
    ax.scatter(
        s.loc[~small_zero, "K"],
        s.loc[~small_zero, "small_time_median"],
        marker="o",
        s=28,
        color=small_color,
        zorder=4,
    )
    ax.scatter(
        s.loc[~mcp_zero, "K"],
        s.loc[~mcp_zero, "mcp_time_median"],
        marker="s",
        s=28,
        color=mcp_color,
        zorder=4,
    )

    # Cross markers only where success rate == 0
    ax.scatter(
        s.loc[small_zero, "K"],
        s.loc[small_zero, "small_time_median"],
        marker="x",
        s=70,
        linewidths=2.0,
        color=small_color,
        label="small-support: failed" if n == n_grid[0] else "_nolegend_",
        zorder=5,
    )
    ax.scatter(
        s.loc[mcp_zero, "K"],
        s.loc[mcp_zero, "mcp_time_median"],
        marker="x",
        s=70,
        linewidths=2.0,
        color=mcp_color,
        label="MCP: failed" if n == n_grid[0] else "_nolegend_",
        zorder=5,
    )

    ax.set_yscale("log")
    ax.set_title(f"n={n}", fontsize=15)
    ax.set_xlabel("K", fontsize=15)
    ax.tick_params(labelsize=15)
    ax.grid(alpha=0.25)
    if n not in [20, 50]:
        ax.set_xticks(range(100, 501, 100))
    else:
        ax.set_xticks(range(100, 301, 100))

axes[0].set_ylabel("Median runtime (s)", fontsize=15)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, -0.08),
    fontsize=15,
)
fig.set_dpi(1200)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

K_grid = [5, 10, 30, 100, 250, 500]
n_grid = [5, 10, 20, 50]

cmap = plt.get_cmap("inferno")
small_color = cmap(0.25)
mcp_color = cmap(0.75)

fig, axes = plt.subplots(
    1,
    len(n_grid),
    figsize=(15, 3.5),
    dpi=150,
    sharey=True,
)

for ax, n in zip(axes, n_grid):
    s = summary[summary["n"] == n].sort_values("K")

    ax.plot(
        s["K"],
        s["small_success_rate"],
        "o-",
        label="small-support" if n == n_grid[0] else "_nolegend_",
        color=small_color,
        linewidth=1.8,
        markersize=4.5,
    )

    ax.plot(
        s["K"],
        s["mcp_success_rate"],
        "s--",
        label="MCP" if n == n_grid[0] else "_nolegend_",
        color=mcp_color,
        linewidth=1.8,
        markersize=4.5,
    )

    ax.set_ylim(-0.05, 1.05)
    ax.set_title(f"n={n}", fontsize=15)
    ax.set_xlabel("K", fontsize=15)
    ax.grid(alpha=0.25)
    if n not in [20, 50]:
        ax.set_xticks(range(100, 501, 100))
    else:
        ax.set_xticks(range(100, 301, 100))
    ax.tick_params(labelsize=15)

axes[0].set_ylabel("Success rate", fontsize=15)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(0.5, -0.08),
    fontsize=15,
)

plt.tight_layout()
plt.show()

In [ ]:
display_cols = [
    "K",
    "n",
    "kappa",
    "tau",
    "reps",
    "problem_size",
    "small_success_rate",
    "mcp_success_rate",
    "small_time_median",
    "mcp_time_median",
    "small_eta_median",
    "mcp_eta_median",
]

summary[display_cols].sort_values(["K", "n"])